# Human tonsil: spatial domains and lymphoid communication

Spatial CITE-seq pairs RNA and ADT at the same observations. This tutorial examines CD40LG–CD40 and CXCL13–CXCR5 signaling in tonsil.

Training is commented out by default. **Run All** uses saved CellSTIC results and recomputes the analyses below. Figures and tables are saved alongside the existing model results under `DATA / "cellstic"`. Run All does not train or modify the saved model.

## 1. Set up the dataset

Start Jupyter from this repository. Use the environment in `environment.yml` and supply the assembled inputs and saved results at the paths below.

In [1]:
from pathlib import Path
import sys
import json
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline/cellstic.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline.analysis import Analysis
from utils.analysis.result_io import load_result, lr_matrices, channel_keys
from utils.analysis import dataset_analysis as da
%matplotlib inline
plt.style.use("default")
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

<conda-env>/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA = ROOT / "paper/data/human_tonsil"
RESULT_PATH = DATA / "cellstic/out/out.h5ad"
OUTPUT = DATA / "cellstic"
OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. Train CellSTIC (optional)

Uncomment both cells to fit the assembled inputs. The pipeline performs normalization, feature extraction, graph construction and training. If explicitly enabled, training writes to the same output location used below and replaces that saved fit.

In [3]:
# from model.train import get_cellstic_config
# from pipeline.cellstic import run_cellstic
# names = ['rna', 'adt']
# inputs = [ad.read_h5ad(DATA / "h5ad" / f"{name}.h5ad") for name in names]
# config = get_cellstic_config('human_tonsil')

In [4]:
# from utils.tools import CellChatDBLoader
# lr_map = {'CXCL13': ['CXCR5'], 'CCL21': ['CCR7'], 'CD40LG': ['CD40'], 'TNFSF13B': ['TNFRSF13B'], 'JAG1': ['NOTCH1']}
# trained = run_cellstic(inputs, config, modality_types=["rna", "adt"],
#                       ligand_receptor_map=lr_map,
#                       cell_chat_db=CellChatDBLoader().get_ligand_receptor_map("human"),
#                       output_path=OUTPUT)
# RESULT_PATH = OUTPUT / "out/out.h5ad"

## 3. Load the saved result

Load the spatial metadata, embedding and predicted graphs. Expression and auxiliary calibration arrays stay on disk.

In [5]:
result = load_result(RESULT_PATH)
display(pd.DataFrame({"observations": [result.n_obs],
                      "domains": [result.obs["cluster"].nunique()],
                      "LR channels": [len(channel_keys(result.obsp, result.uns["cellstic"]))]}))

,observations,domains,LR channels
0,2492,7,5


## 4. Inspect spatial domains

Colors show the saved domain assignments. The spatial visualizer uses the measured coordinates.

In [6]:
from utils.metrics.spatial_viz import SpatialVisualizer
from utils.analysis.figure_export import show_image_row
spatial = ad.AnnData(obs=result.obs.copy(), obsm={"spatial": result.obsm["spatial"]})
spatial.obs["cluster"] = pd.Categorical(result.obs['cluster'].astype(str))
(OUTPUT / "domain").mkdir(parents=True, exist_ok=True)
SpatialVisualizer.generate_spatial_domain_visualization(
    spatial, save_path=OUTPUT / "domain/spatial_domain.svg", size_multiplier=0.85)
show_image_row([OUTPUT / "domain/spatial_domain.svg"])

<conda-env>/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Visualizing 2492 points with 7 domains in grid style
  Calculated min nearest neighbor distance: 1.0000
  Square size (base * 0.85): 1.0200
  Direct visualization: 2492 points, each point gets its own square


Spatial domain visualization (grid style) saved to ./paper/data/human_tonsil/cellstic/domain/spatial_domain.svg


## 5. Map selected ligand–receptor networks

These channels illustrate co-stimulation and lymphoid organization in tonsil. The existing visualizer collapses each directed pair to an undirected display edge and retains at most three incident edges per observation.

In [7]:
from utils.visualization import LigandReceptorSpatialVisualizer
PAIRS = ['CD40LG:CD40', 'CXCL13:CXCR5']
matrices = lr_matrices(result)
selected = {pair: i for i, pair in enumerate(PAIRS)}
LigandReceptorSpatialVisualizer().plot_ligand_receptor_spatial_distribution(
    adata=result, edge_type_map=selected, pos_edge_probs=[matrices[p] for p in PAIRS],
    lr_filter=PAIRS, threshold=0.0, save_path=OUTPUT / "ccc")
figures = [OUTPUT / "ccc" / ("lr_spatial_" + pair.replace(":", "_") + ".svg") for pair in PAIRS]
show_image_row(figures, labels=PAIRS)

Found 2 valid ligand-receptor pairs
